# 02 — Chunking

Explore time-window chunking strategies for transcript snippets.

**Questions to answer:**
- What window size gives the best token count for Pinecone's llama-text-embed-v2? (recommended: 400-500 tokens)
- Do time-based windows cut mid-sentence? How bad is it?
- How much overlap do we need to preserve context at boundaries?

**Design spec:** ~60-90s windows, ~10-15s overlap. Test 60s, 90s, 120s.

**Output:** Production-ready `chunk_transcript()` function → `src/chunking.py`

In [3]:
import json
import tiktoken

### Use this if you want fetch form YouTube
# from pprint import pprint

# # Reuse fetch_transcript from notebook 01 (not imported — this is exploration)
# from youtube_transcript_api import YouTubeTranscriptApi

# ytt_api = YouTubeTranscriptApi()

# # Fetch two videos with different characteristics for testing
# short_video = ytt_api.fetch("dQw4w9WgXcQ")     # 3.5 min, music, sparse text
# long_video = ytt_api.fetch("e-gwvmhyU7A")       # 182 min, podcast, dense speech

# # Convert to plain dicts (same as production function output)
# def to_dicts(transcript):
#     return [{"text": s.text, "start": s.start, "duration": s.duration} for s in transcript.snippets]

# short_snippets = to_dicts(short_video)
# long_snippets = to_dicts(long_video)
### end of fetch directly from YouTube


# Load cached transcripts (avoids YouTube rate limits during development)
with open("../data/test_transcripts.json") as f:
    test_data = json.load(f)

short_snippets = test_data["short"]["snippets"]
long_snippets = test_data["long"]["snippets"]

# Token counter — cl100k_base is close enough for estimating
enc = tiktoken.get_encoding("cl100k_base")
def count_tokens(text: str) -> int:
    return len(enc.encode(text))

print(f"Short video: {len(short_snippets)} snippets, {short_snippets[-1]['start']:.0f}s")
print(f"Long video: {len(long_snippets)} snippets, {long_snippets[-1]['start']:.0f}s")

Short video: 61 snippets, 208s
Long video: 4371 snippets, 10925s


## 1. Naive time-window chunking

Group snippets into fixed time windows. Test 60s, 90s, 120s to see how token counts compare to Pinecone's recommended 400-500 tokens per chunk.

In [4]:
def chunk_by_time(snippets, window_seconds, overlap_seconds=0):
    """Group snippets into time-based chunks. Simple version for exploration."""
    chunks = []
    current_texts = []
    window_start = snippets[0]["start"] if snippets else 0
    
    for s in snippets:
        if s["start"] >= window_start + window_seconds and current_texts:
            text = " ".join(current_texts)
            chunks.append({
                "text": text,
                "start_time": window_start,
                "end_time": s["start"],
                "tokens": count_tokens(text),
            })
            window_start = s["start"] - overlap_seconds
            current_texts = []
        
        current_texts.append(s["text"].replace("\n", " "))
    
    if current_texts:
        text = " ".join(current_texts)
        last = snippets[-1]
        chunks.append({
            "text": text,
            "start_time": window_start,
            "end_time": last["start"] + last["duration"],
            "tokens": count_tokens(text),
        })
    
    return chunks


print(f"{'Window':>8} {'Overlap':>8} {'Chunks':>7} {'Min tok':>8} {'Avg tok':>8} {'Max tok':>8} {'In 400-500':>10}")
print("-" * 70)

for window in [60, 90, 120]:
    for overlap in [0, 10, 15]:
        chunks = chunk_by_time(long_snippets, window, overlap)
        tokens = [c["tokens"] for c in chunks]
        in_range = sum(1 for t in tokens if 400 <= t <= 500)
        pct = in_range / len(tokens) * 100
        avg = sum(tokens) / len(tokens)
        print(f"{window:>6}s {overlap:>6}s {len(chunks):>7} {min(tokens):>8} {avg:>8.0f} {max(tokens):>8} {pct:>9.0f}%")

  Window  Overlap  Chunks  Min tok  Avg tok  Max tok In 400-500
----------------------------------------------------------------------
    60s      0s     178      138      213      293         0%
    60s     10s     213       52      178      251         0%
    60s     15s     235       70      161      229         0%
    90s      0s     120      175      315      381         0%
    90s     10s     134      224      282      355         0%
    90s     15s     143      185      264      332         0%
   120s      0s      90      295      420      511        69%
   120s     10s      98      315      386      473        33%
   120s     15s     103      122      367      458        17%


## 2. Refined chunking — snippet-aware with context carry

Instead of sliding the window start backwards (overlap), carry the last N snippets from the previous chunk into the next one. This preserves complete snippets and adds context without distorting window sizes.

In [5]:
def chunk_by_time_v2(snippets, window_seconds=120, carry_snippets=3):
    """Group snippets into time-based chunks.
    
    - Never splits a snippet
    - Carries last N snippets from previous chunk into the next for context
    - Window is based on the first (non-carried) snippet's start time
    """
    chunks = []
    carry = []
    i = 0
    
    while i < len(snippets):
        current_texts = [s["text"].replace("\n", " ") for s in carry]
        chunk_snippets = list(carry)
        window_start = snippets[i]["start"]
        
        while i < len(snippets) and snippets[i]["start"] < window_start + window_seconds:
            chunk_snippets.append(snippets[i])
            current_texts.append(snippets[i]["text"].replace("\n", " "))
            i += 1
        
        text = " ".join(current_texts)
        chunks.append({
            "text": text,
            "start_time": chunk_snippets[0]["start"],
            "end_time": chunk_snippets[-1]["start"] + chunk_snippets[-1]["duration"],
            "tokens": count_tokens(text),
            "snippet_count": len(chunk_snippets),
        })
        
        # Carry last N snippets (from non-carried portion only)
        non_carry = chunk_snippets[len(carry):]
        carry = non_carry[-carry_snippets:] if carry_snippets > 0 else []
    
    return chunks


# Compare carry strategies on long video
print(f"{'Window':>8} {'Carry':>6} {'Chunks':>7} {'Min tok':>8} {'Avg tok':>8} {'Max tok':>8} {'In 400-500':>10}")
print("-" * 70)

for window in [90, 120]:
    for carry in [0, 2, 3, 5]:
        chunks = chunk_by_time_v2(long_snippets, window, carry)
        tokens = [c["tokens"] for c in chunks]
        in_range = sum(1 for t in tokens if 400 <= t <= 500)
        pct = in_range / len(tokens) * 100
        avg = sum(tokens) / len(tokens)
        print(f"{window:>6}s {carry:>6} {len(chunks):>7} {min(tokens):>8} {avg:>8.0f} {max(tokens):>8} {pct:>9.0f}%")

  Window  Carry  Chunks  Min tok  Avg tok  Max tok In 400-500
----------------------------------------------------------------------
    90s      0     120      175      315      381         0%
    90s      2     120      188      334      399         0%
    90s      3     120      195      343      410         4%
    90s      5     120      208      360      428        12%
   120s      0      90      295      420      511        69%
   120s      2      90      309      439      538        81%
   120s      3      90      315      447      548        81%
   120s      5      90      330      465      563        73%


## 3. Timestamp-aware chunk formatting

Embed snippet timestamps into chunk text so the LLM can reference precise moments.
This decouples retrieval quality (120s chunks) from timestamp precision (snippet-level, ~2-5s).

In [6]:
def format_time(seconds):
    """Convert seconds to H:MM:SS or M:SS display format."""
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = int(seconds % 60)
    if h > 0:
        return f"{h}:{m:02d}:{s:02d}"
    return f"{m}:{s:02d}"


# Show what a timestamped chunk looks like
sample_chunks = chunk_by_time_v2(long_snippets, window_seconds=120, carry_snippets=3)
sample = sample_chunks[5]  # Pick a chunk from the middle

# Rebuild with timestamps from raw snippets
start = sample["start_time"]
end = sample["end_time"]
relevant_snippets = [s for s in long_snippets if s["start"] >= start and s["start"] + s["duration"] <= end + 0.5]

print(f"Chunk 5: {format_time(start)} – {format_time(end)} ({sample['tokens']} tokens)\n")
print("--- Plain text (current) ---")
print(sample["text"][:300])
print("\n--- With timestamps (proposed) ---")
timestamped_lines = []
for s in relevant_snippets[:10]:
    timestamped_lines.append(f"[{format_time(s['start'])}] {s['text'].replace(chr(10), ' ')}")
print("\n".join(timestamped_lines))

Chunk 5: 10:02 – 12:06 (499 tokens)

--- Plain text (current) ---
- Yeah. - Complete. Okay, answer: "Perplexity AI, while impressive, "is not yet a full replacement "for Google for everyday searches. - Yes. - "Here are the key points "based on the provided sources." "Strength of Perplexity AI: "Direct answers, AI-powered summaries, "focus search, user experience."

--- With timestamps (proposed) ---
[10:02] - Yeah. - Complete.
[10:03] Okay, answer:
[10:04] "Perplexity AI, while impressive,
[10:06] "is not yet a full replacement
[10:07] "for Google for everyday searches.
[10:09] - Yes. - "Here are the key points
[10:10] "based on the provided sources."
[10:13] "Strength of Perplexity AI:
[10:14] "Direct answers, AI-powered summaries,
[10:17] "focus search, user experience."


In [7]:
def build_timestamped_text(snippets, start_time, end_time):
    """Build chunk text with inline timestamps."""
    lines = []
    for s in snippets:
        if s["start"] >= start_time - 0.5 and s["start"] <= end_time + 0.5:
            lines.append(f"[{format_time(s['start'])}] {s['text'].replace(chr(10), ' ')}")
    return "\n".join(lines)


# Compare token counts: plain vs timestamped across all chunks
chunks_plain = chunk_by_time_v2(long_snippets, window_seconds=120, carry_snippets=3)

plain_tokens = []
stamped_tokens = []

for c in chunks_plain:
    plain_tokens.append(c["tokens"])
    stamped_text = build_timestamped_text(long_snippets, c["start_time"], c["end_time"])
    stamped_tokens.append(count_tokens(stamped_text))

overhead = [(s - p) / p * 100 for p, s in zip(plain_tokens, stamped_tokens)]

print(f"{'':>12} {'Plain':>10} {'Timestamped':>12} {'Overhead':>10}")
print("-" * 48)
print(f"{'Min tokens':>12} {min(plain_tokens):>10} {min(stamped_tokens):>12} {min(overhead):>9.0f}%")
print(f"{'Avg tokens':>12} {sum(plain_tokens)//len(plain_tokens):>10} {sum(stamped_tokens)//len(stamped_tokens):>12} {sum(overhead)/len(overhead):>9.0f}%")
print(f"{'Max tokens':>12} {max(plain_tokens):>10} {max(stamped_tokens):>12} {max(overhead):>9.0f}%")

                  Plain  Timestamped   Overhead
------------------------------------------------
  Min tokens        315          601        58%
  Avg tokens        447          804        80%
  Max tokens        548          981        97%


## 4. Dual-text strategy

- `text` (plain) → used for embedding, stays in 400-500 token sweet spot
- `text_timestamped` → stored in Pinecone metadata, sent to Claude for answering

This decouples retrieval quality from timestamp precision.

In [8]:
# Pinecone metadata limit: 40KB per vector
# Check if timestamped text fits

import sys

sizes = []
for c in chunks_plain:
    stamped = build_timestamped_text(long_snippets, c["start_time"], c["end_time"])
    sizes.append(len(stamped.encode("utf-8")))

print(f"Timestamped text size (bytes):")
print(f"  Min: {min(sizes):,}")
print(f"  Avg: {sum(sizes)//len(sizes):,}")
print(f"  Max: {max(sizes):,}")
print(f"  Pinecone limit: 40,960")
print(f"  Fits: {'✅' if max(sizes) < 40960 else '❌'}")

Timestamped text size (bytes):
  Min: 1,875
  Avg: 2,490
  Max: 2,785
  Pinecone limit: 40,960
  Fits: ✅


## 5. Final comparison — lock in parameters

120s window, 3 carry snippets gave the best token distribution. Confirm with both videos.

In [9]:
for label, snippets in [("short (3.5 min)", short_snippets), ("long (182 min)", long_snippets)]:
    chunks = chunk_by_time_v2(snippets, window_seconds=120, carry_snippets=3)
    tokens = [c["tokens"] for c in chunks]
    in_range = sum(1 for t in tokens if 400 <= t <= 500)
    pct = in_range / len(tokens) * 100 if tokens else 0
    avg = sum(tokens) / len(tokens) if tokens else 0
    
    print(f"\n{label}:")
    print(f"  Chunks: {len(chunks)}")
    print(f"  Tokens: min={min(tokens)}, avg={avg:.0f}, max={max(tokens)}")
    print(f"  In 400-500 range: {pct:.0f}%")
    print(f"  First chunk: {chunks[0]['text'][:80]!r}")
    print(f"  Timestamps: {format_time(chunks[0]['start_time'])} – {format_time(chunks[-1]['end_time'])}")


short (3.5 min):
  Chunks: 2
  Tokens: min=335, avg=340, max=344
  In 400-500 range: 0%
  First chunk: "[♪♪♪] ♪ We're no strangers to love ♪ ♪ You know the rules and so do I ♪ ♪ A full"
  Timestamps: 0:01 – 3:31

long (182 min):
  Chunks: 90
  Tokens: min=315, avg=447, max=548
  In 400-500 range: 81%
  First chunk: '- Can you have a conversation with an AI where it feels like you talk to Einstei'
  Timestamps: 0:00 – 3:02:06


## 6. Decision

| Parameter | Value | Reasoning |
|---|---|---|
| Window size | 120s | Best token fit (avg 447, 81% in 400-500 range) |
| Carry snippets | 3 | Adds ~15-20s of context at boundaries |
| Timestamp approach | Dual-text | Plain for embedding, timestamped for Claude |
| Timestamp precision | Snippet-level (~2-5s) | Well within the 30s target |

Short videos will have below-target token counts — acceptable since they have less content to retrieve from anyway.

## 7. Production function

In [10]:
# @export src/chunking.py
def format_time(seconds: float) -> str:
    """Convert seconds to H:MM:SS or M:SS display format."""
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = int(seconds % 60)
    if h > 0:
        return f"{h}:{m:02d}:{s:02d}"
    return f"{m}:{s:02d}"


def chunk_transcript(
    snippets: list[dict],
    video_id: str,
    window_seconds: int = 120,
    carry_snippets: int = 3,
) -> list[dict]:
    """Chunk transcript snippets into time-based windows.

    Args:
        snippets: list of {"text": str, "start": float, "duration": float}
        video_id: YouTube video ID (for generating URLs)
        window_seconds: target chunk duration in seconds
        carry_snippets: number of snippets to carry from previous chunk for context

    Returns:
        list of chunk dicts with:
            - text: plain text (for embedding)
            - text_timestamped: text with inline timestamps (for LLM context)
            - start_time: float (seconds)
            - end_time: float (seconds)
            - start_display: str (human readable)
            - end_display: str (human readable)
            - chunk_index: int
            - video_url: str (deep link to timestamp)
    """
    if not snippets:
        return []

    chunks = []
    carry = []
    i = 0

    while i < len(snippets):
        chunk_snippets = list(carry)
        window_start = snippets[i]["start"]

        while i < len(snippets) and snippets[i]["start"] < window_start + window_seconds:
            chunk_snippets.append(snippets[i])
            i += 1

        # Plain text (for embedding)
        plain_parts = [s["text"].replace("\n", " ") for s in chunk_snippets]
        text = " ".join(plain_parts)

        # Timestamped text (for LLM)
        stamped_lines = [
            f"[{format_time(s['start'])}] {s['text'].replace(chr(10), ' ')}"
            for s in chunk_snippets
        ]
        text_timestamped = "\n".join(stamped_lines)

        start_time = chunk_snippets[0]["start"]
        end_time = chunk_snippets[-1]["start"] + chunk_snippets[-1]["duration"]

        chunks.append({
            "text": text,
            "text_timestamped": text_timestamped,
            "start_time": start_time,
            "end_time": end_time,
            "start_display": format_time(start_time),
            "end_display": format_time(end_time),
            "chunk_index": len(chunks),
            "video_url": f"https://youtu.be/{video_id}?t={int(start_time)}",
        })

        # Carry last N snippets from non-carried portion
        non_carry = chunk_snippets[len(carry):]
        carry = non_carry[-carry_snippets:] if carry_snippets > 0 else []

    return chunks

In [11]:
chunks = chunk_transcript(long_snippets, "e-gwvmhyU7A")

print(f"Total chunks: {len(chunks)}")
print(f"\n--- Chunk 5 (sample) ---")
c = chunks[5]
for key in ["chunk_index", "start_display", "end_display", "video_url"]:
    print(f"  {key}: {c[key]}")
print(f"  text tokens: {count_tokens(c['text'])}")
print(f"  text (first 120 chars): {c['text'][:120]!r}")
print(f"\n  timestamped (first 5 lines):")
for line in c["text_timestamped"].split("\n")[:5]:
    print(f"    {line}")

# Verify all chunks have required keys
required = {"text", "text_timestamped", "start_time", "end_time", "start_display", "end_display", "chunk_index", "video_url"}
for i, c in enumerate(chunks):
    missing = required - set(c.keys())
    if missing:
        print(f"  ❌ Chunk {i} missing: {missing}")
        break
else:
    print(f"\n✅ All {len(chunks)} chunks have required keys")

# Verify chunk ordering
for i in range(len(chunks) - 1):
    if chunks[i + 1]["start_time"] <= chunks[i]["start_time"]:
        print(f"  ❌ Chunk {i+1} starts before chunk {i}")
        break
else:
    print(f"✅ All chunks in chronological order")

Total chunks: 90

--- Chunk 5 (sample) ---
  chunk_index: 5
  start_display: 10:02
  end_display: 12:06
  video_url: https://youtu.be/e-gwvmhyU7A?t=602
  text tokens: 499
  text (first 120 chars): '- Yeah. - Complete. Okay, answer: "Perplexity AI, while impressive, "is not yet a full replacement "for Google for every'

  timestamped (first 5 lines):
    [10:02] - Yeah. - Complete.
    [10:03] Okay, answer:
    [10:04] "Perplexity AI, while impressive,
    [10:06] "is not yet a full replacement
    [10:07] "for Google for everyday searches.

✅ All 90 chunks have required keys
✅ All chunks in chronological order


## Summary

**What we built:**
- `format_time(seconds)` — converts float seconds to human-readable `M:SS` or `H:MM:SS`
- `chunk_transcript(snippets, video_id)` — produces chunks with dual text (plain + timestamped), metadata, and deep-link URLs

**Production cells tagged:** 1 cell → `src/chunking.py`

**Key decisions:**

| Decision | Choice | Reasoning |
|---|---|---|
| Window size | 120s | avg 447 tokens, 81% in Pinecone's 400-500 sweet spot |
| Context carry | 3 snippets | ~15-20s overlap, preserves context at boundaries |
| Text strategy | Dual (plain + timestamped) | Plain for embedding (good retrieval), timestamped for Claude (precise timestamps) |
| Timestamp precision | Snippet-level (~2-5s) | Well within the 30s-before target |
| Snippet integrity | Never split | Chunks always end/start at snippet boundaries |

**Token overhead analysis:**
- Timestamped text is ~80% larger than plain text
- Stored in Pinecone metadata (max 2.8KB vs 40KB limit) — no issue
- Never used for embedding — no impact on retrieval quality

**Tested on:**
- Short video (3.5 min, 61 snippets) → 2 chunks, 340 avg tokens (below range, expected)
- Long podcast (182 min, 4371 snippets) → 90 chunks, 447 avg tokens, 81% in range

**Next:** Notebook 03 — Pinecone Operations (embed chunks, upsert, test retrieval quality)